# Retail Sales Forecasting Pipeline

**Objective**: Achieve WAPE < 0.3 using AutoGluon with Global Reindexing Strategy.

### Key Features:
1. **Global Reindexing**: Fixes the "plunge to zero" by ensuring 100% data coverage for all items.
2. **AutoGluon Ensemble**: Leverages DeepAR, Chronos, and Tabular models.
3. **Robust Evaluation**: Direct comparison against Ground Truth data.

In [10]:
!pip -q install autogluon.timeseries "torch<2.10" torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 53.4 MB/s eta 0:00:00


In [3]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import shutil

print("Libraries Loaded")

Libraries Loaded


## 1. Load Raw Data

In [4]:
train_path = 'train_set_100_missing15.csv'
test_path  = 'test_submission.csv'
gt_path    = 'test_ground_truth.csv'

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)
gt    = pd.read_csv(gt_path) if os.path.exists(gt_path) else None

print(f"Train Rows: {len(train)}")

Train Rows: 493650


## 2. Global Reindexing Strategy
We reconstruct the timeline for every product to ensure there are no missing dates (NaN) that cause the model to drop products.

In [5]:
train = train.dropna(subset=['store_id', 'product_id', 'dt'])
train['item_id'] = train['store_id'].astype(int).astype(str) + '_' + train['product_id'].astype(int).astype(str)
train['timestamp'] = pd.to_datetime(train['dt'])

min_dt = train['timestamp'].min()
max_dt = train['timestamp'].max()
all_dates = pd.date_range(start=min_dt, end=max_dt, freq='D')

print(f"Global Range: {min_dt.date()} to {max_dt.date()}")

processed_dfs = []
item_groups = train.groupby('item_id')
items = train['item_id'].unique()

for i, item_id in enumerate(items):
    group = item_groups.get_group(item_id).drop_duplicates(subset='timestamp')
    group = group.set_index('timestamp').reindex(all_dates)
    group['item_id'] = item_id
    group['sale_amount'] = group['sale_amount'].interpolate(method='linear', limit_direction='both').fillna(0)

    # Simple Covariate Fill
    cov_cols = ['discount', 'holiday_flag', 'activity_flag', 'precpt']
    for col in cov_cols:
        if col in group.columns: group[col] = group[col].ffill().bfill().fillna(0)

    processed_dfs.append(group)

train_cleaned = pd.concat(processed_dfs).reset_index().rename(columns={'index': 'timestamp'})
print("Reindexing Done")

Global Range: 2024-03-28 to 2024-06-25
Reindexing Done


## 3. Train AutoGluon Model

In [11]:
item_static = train_cleaned.groupby('item_id')['sale_amount'].agg(['mean', 'std', 'max']).reset_index()

train_tsdf = TimeSeriesDataFrame.from_data_frame(
    train_cleaned,
    id_column='item_id',
    timestamp_column='timestamp'
)
train_tsdf.static_features = item_static.set_index('item_id')

predictor = TimeSeriesPredictor(
    target='sale_amount',
    prediction_length=7,
    freq='D',
    eval_metric='WAPE',
    path='output/model/autogluon_v8',
).fit(
    train_tsdf,
    time_limit=600,
    presets='best_quality',
)

Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/content/output/model/autogluon_v8'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
GPU Memory:         
Total GPU Memory:   Free: 0.00 GB, Allocated: 0.00 GB, Total: 0.00 GB
GPU Count:          0
Memory Avail:       9.97 GB / 12.67 GB (78.7%)
Disk Space Avail:   187.94 GB / 225.83 GB (83.2%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'freq': 'D',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 'auto',
 'prediction_length': 7,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 'auto',
 'ref

## 4. Evaluation (WAPE)

In [12]:
import os
import pandas as pd
import numpy as np

# Ensure the output directory exists
output_dir = 'output/model/'
os.makedirs(output_dir, exist_ok=True)

# Assuming 'predictor' and 'train_tsdf' are available from previous cells.
# Generate predictions
predictions = predictor.predict(train_tsdf)

# Convert predictions to the submission format
# predictions is a TimeSeriesDataFrame with index as item_id and timestamp, and 'mean' column
submission_df = predictions.reset_index()
submission_df = submission_df[['item_id', 'timestamp', 'mean']]
submission_df = submission_df.rename(columns={'timestamp': 'dt', 'mean': 'sale_amount'})

# Split item_id back into store_id and product_id for the submission file
# Example: '67_768' -> store_id=67, product_id=768
submission_df['store_id'] = submission_df['item_id'].apply(lambda x: int(x.split('_')[0]))
submission_df['product_id'] = submission_df['item_id'].apply(lambda x: int(x.split('_')[1]))

# Reorder columns to match potential submission requirements (dt, store_id, product_id, sale_amount)
submission_df = submission_df[['dt', 'store_id', 'product_id', 'sale_amount']]

# Save the predictions to the specified path
submission_file_path = os.path.join(output_dir, 'submission_v8.csv')
submission_df.to_csv(submission_file_path, index=False)

# Now, the rest of the original code can execute
if gt is not None:
    sub = pd.read_csv(submission_file_path) # Read the newly created submission file
    gt['item_id'] = gt['store_id'].astype(int).astype(str) + '_' + gt['product_id'].astype(int).astype(str)
    sub['item_id'] = sub['store_id'].astype(int).astype(str) + '_' + sub['product_id'].astype(int).astype(str)

    eval_df = gt.merge(sub[['item_id', 'dt', 'sale_amount']], on=['item_id', 'dt'], how='inner')
    abs_err = np.abs(eval_df['sale_amount_x'] - eval_df['sale_amount_y'])
    wape = abs_err.sum() / eval_df['sale_amount_x'].sum()
    print(f"Final Ground Truth WAPE: {wape:.4f}")

Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


Final Ground Truth WAPE: 0.3831
